In [1]:
import lseg.data as ld
import pandas as pd
from tqdm import tqdm
from datetime import datetime
ld.open_session()
print(f"{ld.__version__}")

2.0.1


In [3]:
dates = pd.date_range(start='2016-06-01', end='2026-02-01', freq='M').strftime('%Y-%m-%d').tolist()

master_data = []

print(f"Starting data extraction for {len(dates)} months...")

for date in tqdm(dates):
    try:
        # Step A: Get constituents
        constituents_df = ld.get_data(
            universe=['.SPX'],
            fields=['TR.IndexConstituentRIC'],
            parameters={'SDate': date}
        )
        
        ric_list = constituents_df['Constituent RIC'].tolist()
        
        # Step B: Loop through RIC list in batches
        for i in range(0, len(ric_list), 50):
            batch = ric_list[i : i + 50]
            
            # Request data
            batch_data = ld.get_data(
                universe=batch,
                fields=['TR.PriceClose','TR.CompanyMarketCap'],
                parameters={'SDate': date}
            )
            
            # Add the date and append directly to the master list
            batch_data['Date'] = date
            master_data.append(batch_data)
            
    except Exception as e:
        print(f"Error fetching data for {date}: {e}")

# Step C: Flattened list of DataFrames is ready to concat
df_final = pd.concat(master_data, ignore_index=True)
df_final.to_csv("SP500_Historical_Monthly.csv", index=False)

Starting data extraction for 116 months...


100%|██████████| 116/116 [25:52<00:00, 13.39s/it]


In [2]:
df = pd.read_csv("STOXX600_Historical_Monthly.csv")

uni = df['Instrument'].unique().tolist()

sectors_stoxx = ld.get_data(
            universe = uni,
            fields = ['TR.TRBCIndustry',
                     'TR.TRBCActivity'])

sectors_stoxx.to_csv("sectors_stoxx.csv", index = False)

In [9]:
df = pd.read_csv("STOXX600_Historical_Monthly.csv")

uni = df['Instrument'].unique().tolist()

escore_stoxx = ld.get_data(
            universe = uni,
            fields = ['TR.ESGPlusCompleteScore', 'TR.EnvironmentalPillarESGScore'],
            parameters = {'SDate': '2016-01-01', 'EDate':'2025-01-01', 'Frq':'Y'})

escore_stoxx.to_csv("escore_stoxx.csv", index = False)

  Instrument       Date  1 Month Total Return
0         DE 2018-06-29             -8.408709


Building unique constituent list...
Checking constituents for 2016-12-31...
Checking constituents for 2017-12-31...
Checking constituents for 2018-12-31...
Checking constituents for 2019-12-31...
Checking constituents for 2020-12-31...
Checking constituents for 2021-12-31...
Checking constituents for 2022-12-31...
Checking constituents for 2023-12-31...
Checking constituents for 2024-12-31...
Checking constituents for 2025-12-31...
Checking constituents for 2026-03-05...
Done! Created a unique list of 676 instruments.



Starting History Extraction...
Breaking request into batches of 100 to prevent timeouts...


Fetching Data Batches: 100%|██████████| 7/7 [00:56<00:00,  8.09s/it]



History Extraction Complete.
First 5 rows of data:
           Instrument  Close Price  Company Market Cap  \
Date                                                     
2016-01-29        A.N        37.65       12506234008.5   
2016-01-29      PGR.N        31.25      18267007656.25   
2016-01-29      AZO.N       767.39      23181551941.34   
2016-01-29       PH.N        97.16      13126566284.16   
2016-01-29      PHM.N        16.76       5851538516.68   

                                  TRBC Industry Name TRBC Industry Code  \
Date                                                                      
2016-01-29   Advanced Medical Equipment & Technology           56101010   
2016-01-29             Property & Casualty Insurance           55301020   
2016-01-29  Auto Vehicles, Parts & Service Retailers           53403010   
2016-01-29          Industrial Machinery & Equipment           52102010   
2016-01-29                              Homebuilding           53203010   

               

Building unique constituent list...
Checking constituents for 2016-12-31...
Checking constituents for 2017-12-31...
Checking constituents for 2018-12-31...
Checking constituents for 2019-12-31...
Checking constituents for 2020-12-31...
Checking constituents for 2021-12-31...
Checking constituents for 2022-12-31...
Checking constituents for 2023-12-31...
Checking constituents for 2024-12-31...
Checking constituents for 2025-12-31...
Checking constituents for 2026-03-05...
Error fetching data for 2026: Unable to resolve and collect data for all requested identifiers and fields. Requested universes: ['.STOXX']. Requested fields: ['TR.INDEXCONSTITUENTRIC']
Done! Created a unique list of 916 instruments.



Starting History Extraction...
Breaking request into batches of 100 to prevent timeouts...


Fetching Data Batches: 100%|██████████| 10/10 [01:33<00:00,  9.37s/it]



History Extraction Complete.
First 5 rows of data:
              Instrument  Close Price  Company Market Cap  \
Date                                                        
2016-01-13         HBR.L        380.0         97054101.59   
2016-01-29  1COVG.DE^L25         30.4    6155999999.99999   
2016-01-29       RRTL.DE        74.52       11454278996.0   
2016-01-29        RS1R.L        209.1    921528421.892994   
2016-01-29     RSA.L^F21        416.4      4235037182.088   

                              TRBC Industry Name TRBC Industry Code  \
Date                                                                  
2016-01-13  Oil & Gas Exploration and Production           50102020   
2016-01-29                   Commodity Chemicals           51101010   
2016-01-29                          Broadcasting           53302020   
2016-01-29      Industrial Machinery & Equipment           52102010   
2016-01-29         Property & Casualty Insurance           55301020   

                      

In [6]:
ld.close_session()